In [ ]:
# os tools
import sys
import os
import os.path
import requests
import time
import urllib3
import shutil
from tqdm import tqdm

# data tools
import numpy                 as np
import pandas                as pd
import xarray                as xr
from   datetime              import date, datetime, timedelta                 # for saving figures with today's date
import datetime
import scipy
from   scipy.stats           import kruskal              # for boxenplot stats
from   scipy.stats           import mannwhitneyu         # for split violin plot stats
import gsw


# for all plots
import matplotlib
import matplotlib.pyplot     as plt                      # needed to make map setup
import matplotlib.colors     as colors
from   matplotlib.ticker     import EngFormatter         # for degree symbol in axis
from   cmocean               import cm as cmo
import seaborn               as     sns

# for map
import shapefile
import cartopy                                           # to make map
import matplotlib.path       as     mpath                # to draw circle for map
import cartopy.crs           as     ccrs                 # for map projection
import cartopy.feature       as     cfeature             # to add land features to map
# from   xhistogram.xarray     import histogram            # for map histogram
# from   mycolorpy             import colorlist as mcp     # to get n colors list
import pyproj  
import geopandas             as     gpd                  # for adding shapefiles of frontal zones 
from   osgeo                 import gdal
# import scikit_posthocs       as     sp                   # for stats


xr.set_options(display_expand_attrs = False)

In [ ]:
# Custom modules
import mod_cremas as crx 
import mod_ocean as myocn
import mod_argo 

from importlib import reload
import mod_plotting as myplt

plt.rcParams.update(myplt.my_params(size=12))

import shapefile
so_fronts = shapefile.Reader('./shapefiles/fronts/so_fronts.shp') 
stf_mod   = shapefile.Reader('./shapefiles/fronts/stf_mod/stf_mod.shp')

stf  = stf_mod.shape(0).points
saf  = so_fronts.shape(1).points
pf   = so_fronts.shape(2).points
sacc = so_fronts.shape(3).points
sie  = so_fronts.shape(4).points

max_latitude:          float = -30
add_gridlines:         bool  = True
color_land:            bool  = False
land_edgecolor:        str   = 'grey'
land_facecolor:        str   = 'grey'
fontsize:              float = 10
map_facecolor:         str   = 'white'
coast_linewidth:       float = 0.3
gridlines_linewidth:   float = 0.5
girdlines_color:       str   = 'grey'
gridlines_alpha:       float = 0.5
longitude_label_color: str   = 'grey'
latitude_label_color:  str   = 'grey'



In [ ]:
coreDS = xr.open_dataset('/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CREMAS/working-vars/argo/coreDS_qc_interp_2014-2023_acc20250218.nc')

In [ ]:
coreDS_byProfile = coreDS.mean(dim='profid')

In [ ]:
# Plot time series of CT sections for a random float.

# reload(crx)
# reload(crplot)
# reload(mod_argo)
anyfloat, anywmo = mod_argo.get_wmo_DS(coreDS, wmo=None) #wmo=None gives random
anyfloat = anyfloat.to_dataframe().reset_index()

fig, axs = mod_argo.plot_TS_diagnostics(anyfloat, figsize=(12,8), dot_small=1, dot_medium = 6, dot_large=40)


In [ ]:
# def plot_SO_float_locations(argoDS, figsize=(12,8), dot_size = 2):
#     """ 
#     Plot locations of all floats in a dataset.
#     @param: argoDS: xr Dataset with 'latitude', 'longitude' coordinates
#     @return: fig, ax
#     """

# Choose projection
map_proj = ccrs.SouthPolarStereo()
# map_proj = ccrs.PlateCarree()
fsize = (18,18)
color_by_month = False

data = core_INDEX
# === START FXN

fig  = plt.figure(figsize=fsize, dpi=300, layout='tight') 
ax1  = plt.subplot(projection = map_proj)

# Set up plot axes
myplt.setup_SO_axes(ax1, fig , 
                            add_gridlines         = False, 
                            color_land            = True,
                            land_facecolor        = land_facecolor,
                            land_edgecolor        = land_edgecolor,
                            fontsize              = 14,
                            max_latitude         = -35,
                            map_facecolor         = 'white',
                            coast_linewidth       = coast_linewidth)


# data = expo_dict[expocode]
ax1.scatter(data.longitude, data.latitude, c='crimson', alpha=0.2, s=2, transform=ccrs.PlateCarree(), label='')
# ax1.scatter(core_index.longitude, core_index.latitude, c='darkorange', s=8, transform=ccrs.PlateCarree(), label='core Argo')
# ax1.scatter(bgc_index.longitude, bgc_index.latitude, c='c', s=8, transform=ccrs.PlateCarree(), label='bgc Argo')
ax1.legend()

if color_by_month:
    # Get distinct colors for months (assuming 12 months)
    palette = sns.color_palette("Spectral", 12)

    # Normalize month values to use in scatter colormap
    norm = plt.Normalize(1, 12)  
    sm = plt.cm.ScalarMappable(cmap="Spectral", norm=norm)

    # for idf in [annual_index['2015']]:
        # colors = [palette[m - 1] for m in idf.month]
        # ax1.scatter(idf.longitude, idf.latitude, c=colors, s=3, transform=ccrs.PlateCarree())
    # ax1.legend()

    cbar = plt.colorbar(sm, ax=ax1, orientation="horizontal", fraction=0.02, pad=0.1)
    cbar.set_label("Month")
    cbar.set_ticks(np.arange(1, 13))
    cbar.set_ticklabels([str(i) for i in range(1, 13)])


### Add front and sea ice edge
stf_patch  = plt.Polygon(stf,  fill=False, edgecolor='grey',   zorder=15)
saf_patch  = plt.Polygon(saf,  fill=False, edgecolor='grey',   zorder=14)
pf_patch   = plt.Polygon(pf,   fill=False, edgecolor='grey',    zorder=13)
sacc_patch = plt.Polygon(sacc, fill=False, edgecolor='grey',  zorder=12)
sie_patch  = plt.Polygon(sie,  fill=True,  edgecolor='grey',   zorder=0,  facecolor='darkgrey', alpha=0.4)

ax1.add_patch(stf_patch)
ax1.add_patch(saf_patch)
ax1.add_patch(pf_patch)
ax1.add_patch(sacc_patch)
ax1.add_patch(sie_patch)
